In [ ]:
from pathlib import Path
import json, time, platform, inspect, types, datetime, hashlib, shutil
import numpy as np
import scipy
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

OUT = Path.cwd() / "combined_online_observer_output"
OUT.mkdir(parents=True, exist_ok=True)
RESUME = True  # Only this notebook's compatible, newly generated checkpoints.
ENVIRONMENT = dict(python=platform.python_version(), numpy=np.__version__,
                   scipy=scipy.__version__, pandas=pd.__version__, matplotlib=matplotlib.__version__)
print("Output directory:", OUT.resolve())
print("Stage I uses 60 prefix samples; Stage II starts at global sample 59 (no prefix replay).")
print("Environment:", ENVIRONMENT)

In [ ]:

from dataclasses import dataclass, asdict
import numpy as np
from scipy.special import gammaln, digamma

class NumericalFailure(RuntimeError):
    pass

@dataclass(frozen=True)
class Config:
    substeps: int = 2
    gain: float = 0.35
    learning_rate: float = 0.03
    leakage: float = 0.002
    qmin: float = 0.01
    qmax: float = 0.98
    max_q_step: float = 0.02
    newton_tol: float = 1e-10
    max_newton: int = 35
    state_limit: float = 1e4  # Reject, never clip states.

def field(x, a, b):
    u,v,w=x
    return np.array([v,w,-u-b*(1-u*u)*v-a*w])

def jacobian(x,a,b):
    u,v,w=x
    return np.array([[0.,1.,0.],[0.,0.,1.],[-1+2*b*u*v,-b*(1-u*u),-a]])

def l1_kernel(q,n,h):
    if not (0<q<=1 and h>0 and n>=1):
        raise ValueError('Require 0<q<=1, h>0, n>=1')
    j=np.arange(1,n,dtype=float)
    # expm1 avoids subtracting almost equal powers as q approaches 1.
    logj=np.log(j); logratio=np.log1p(1/j)
    base=np.exp((1-q)*logj)
    weights=base*np.expm1((1-q)*logratio)
    dweights=-logj*weights-base*np.exp((1-q)*logratio)*logratio
    c=np.exp(-q*np.log(h)-gammaln(2-q))
    dc=c*(-np.log(h)+digamma(2-q))
    return weights,dweights,c,dc

def implicit_step(history,q,h,a,b,cfg=Config(),sensitivity_history=None):
    """Solve L1 nonlinear equations by residual-controlled damped Newton.

    sensitivity_history differentiates all states depending on the CURRENT
    interval's q, including preceding substeps. Earlier intervals are fixed.
    Returns state, dy/dq vector, infinity-norm residual, Newton iterations.
    """
    hist=np.asarray(history,dtype=float);n=len(hist)
    if hist.ndim!=2 or hist.shape[1]!=3 or not np.all(np.isfinite(hist)):
        raise NumericalFailure('Invalid state history')
    w,dw,c,dc=l1_kernel(q,n,h)
    delta=np.diff(hist,axis=0)[::-1]
    memory=w@delta
    dmemory=dw@delta
    rhs=hist[-1]-memory
    x=hist[-1].copy()
    def residual(v): return c*(v-rhs)-field(v,a,b)
    r=residual(x)
    for it in range(cfg.max_newton+1):
        norm=float(np.linalg.norm(r,np.inf))
        if np.isfinite(norm) and norm<=cfg.newton_tol:
            break
        if it==cfg.max_newton or not np.isfinite(norm):
            raise NumericalFailure('Newton residual did not converge')
        matrix=c*np.eye(3)-jacobian(x,a,b)
        try: step=np.linalg.solve(matrix,-r)
        except np.linalg.LinAlgError as e: raise NumericalFailure('Singular Newton matrix') from e
        accepted=False
        for k in range(22):
            cand=x+(0.5**k)*step
            if not np.all(np.isfinite(cand)) or np.max(np.abs(cand))>cfg.state_limit:
                continue
            rr=residual(cand)
            if np.all(np.isfinite(rr)) and np.linalg.norm(rr,np.inf)<norm:
                x,r=cand,rr;accepted=True;break
        if not accepted: raise NumericalFailure('Newton line search failed')
    if np.max(np.abs(x))>cfg.state_limit: raise NumericalFailure('State exceeded rejection bound')
    ds_prev=np.zeros(3)
    if sensitivity_history is not None:
        sh=np.asarray(sensitivity_history,dtype=float)
        if sh.shape!=hist.shape: raise ValueError('Sensitivity history shape mismatch')
        ds_prev=sh[-1]
        dmemory=dmemory+w@np.diff(sh,axis=0)[::-1]
    partial=dc*(x-rhs)+c*(-ds_prev+dmemory)
    try: sensitivity=np.linalg.solve(c*np.eye(3)-jacobian(x,a,b),-partial)
    except np.linalg.LinAlgError as e: raise NumericalFailure('Singular sensitivity matrix') from e
    if not np.all(np.isfinite(sensitivity)): raise NumericalFailure('Nonfinite sensitivity')
    return x,sensitivity,norm,it


def metrics(truth,prediction):
    e=np.asarray(prediction)-np.asarray(truth)
    if not np.all(np.isfinite(e)): raise ValueError('Nonfinite metric input')
    return dict(mse=float(np.mean(e*e)),rmse=float(np.sqrt(np.mean(e*e))),mae=float(np.mean(np.abs(e))))

In [ ]:

import json,time
from pathlib import Path
from dataclasses import replace
import numpy as np
from scipy.optimize import least_squares
from scipy.interpolate import BSpline

def interpolation_basis(n,k,substeps=1):
    # Clamped cubic B-spline: partition of unity preserves order bounds;
    # C2 continuity in the interior. Coordinates are sample fraction, not h.
    if any(not isinstance(v, (int, np.integer)) or isinstance(v, bool) for v in (n,k,substeps)) or n<2 or k<1 or substeps<1:
        raise ValueError("Require integer n>=2, k>=1, substeps>=1")
    t=np.linspace(0,1,(n-1)*substeps+1)
    degree=min(3,k-1)
    knots=np.r_[np.zeros(degree+1),np.linspace(0,1,k-degree+1)[1:-1],np.ones(degree+1)]
    return BSpline.design_matrix(t,knots,degree).toarray()

def simulate_tangent(theta,initial_y,basis,substeps=1,with_jac=True):
    theta=np.asarray(theta,dtype=float);basis=np.asarray(basis,dtype=float)
    if theta.ndim!=1 or len(theta)<6 or not np.all(np.isfinite(theta)) or not np.isfinite(initial_y):
        raise ValueError("Finite parameter vector and initial observation required")
    if not isinstance(substeps,(int,np.integer)) or isinstance(substeps,bool) or substeps<1:
        raise ValueError("Positive integer substeps required")
    if basis.ndim!=2 or basis.shape[1]!=len(theta)-5 or len(basis)<2 or (len(basis)-1)%substeps or not np.all(np.isfinite(basis)):
        raise ValueError("Basis dimensions must match parameters and observation grid")
    a,b,x0,z0,log_h=theta[:5];qk=np.asarray(theta[5:]);h=np.exp(log_h)/substeps
    qs=basis@qk;N=len(qs);P=len(theta)
    if not np.isfinite(h) or h<=0 or np.any(qs<=0) or np.any(qs>1):
        raise ValueError("Require finite positive time step and 0<q<=1")
    hist=np.empty((N,3));hist[0]=[x0,initial_y,z0]
    tang=np.zeros((N,3,P));tang[0,0,2]=1;tang[0,2,3]=1
    max_res=0.
    for n in range(1,N):
        x,_,res,_=implicit_step(hist[:n],float(qs[n]),h,a,b)
        hist[n]=x;max_res=max(max_res,res)
        if not with_jac:continue
        w,dw,c,dc=l1_kernel(float(qs[n]),n,h)
        dhist=np.diff(hist[:n],axis=0)[::-1]
        delta=x-hist[n-1]+w@dhist
        q_direction=np.zeros(P);q_direction[5:]=basis[n]
        partial=np.outer(dc*delta+c*(dw@dhist),q_direction)
        partial[:,4]+=-qs[n]*c*delta  # derivative wrt log(time scale)
        memory_s=np.tensordot(w,np.diff(tang[:n],axis=0)[::-1],axes=(0,0))
        partial+=c*(-tang[n-1]+memory_s)
        partial[2,0]+=x[2]
        partial[2,1]+=(1-x[0]**2)*x[1]
        try: tang[n]=np.linalg.solve(c*np.eye(3)-jacobian(x,a,b),-partial)
        except np.linalg.LinAlgError as exc: raise NumericalFailure('Singular full-history tangent matrix') from exc
        if not np.all(np.isfinite(tang[n])):raise NumericalFailure('Nonfinite tangent')
    return hist,tang,max_res

In [ ]:

import hashlib,json
from pathlib import Path
import numpy as np
from scipy.optimize import least_squares
from scipy.special import comb

VERSION='research-final-v3'
def fingerprint(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,allow_nan=False).encode()).hexdigest()
def basis_matrix(n,k,substeps=1,kind='bspline'):
    if kind=='bspline':return interpolation_basis(n,k,substeps)
    if kind!='bernstein':raise ValueError('Unknown basis family')
    # Both families are nonnegative and sum to one, preserving q bounds.
    interpolation_basis(n,k,substeps)  # shared dimensional input checks
    t=np.linspace(0.,1.,(n-1)*substeps+1)[:,None];j=np.arange(k)
    return comb(k-1,j)*t**j*(1-t)**(k-1-j)

def extended_basis(n_train,n_total,k,substeps,kind='bspline'):
    if n_total<n_train:raise ValueError('Total length must include all training observations')
    train=basis_matrix(n_train,k,substeps,kind)
    # The entire old basis, including the endpoint, is exactly preserved.
    return np.vstack([train,np.repeat(train[-1:,:],(n_total-n_train)*substeps,axis=0)])
def parameter_bounds(k):
    return np.r_[-10.,-10.,-3.,-3.,np.log(.03),np.full(k,.03)],np.r_[10.,10,3.,3.,np.log(3.),np.full(k,.98)]
def make_objective(y,k,grids,regularization,normalization=None,basis_kind='bspline',quartic_weight=0.):
    y=np.asarray(y,dtype=float)
    if y.ndim!=1 or len(y)<3 or not np.all(np.isfinite(y)):raise ValueError('Finite one-dimensional observations required')
    if len(grids)!=2 or grids[1]!=2*grids[0]:raise ValueError('Require two successive factor-two meshes')
    if not np.isfinite(regularization) or regularization<0:raise ValueError('Invalid regularization')
    if not np.isfinite(quartic_weight) or quartic_weight<0:raise ValueError('Invalid quartic weight')
    mu,sd=(float(y.mean()),float(y.std())) if normalization is None else map(float,normalization)
    if not np.isfinite(mu) or not np.isfinite(sd) or sd<=1e-12:raise ValueError('Zero/invalid observation scale; order is not identifiable')
    n=len(y);yn=(y-mu)/sd;bs={s:basis_matrix(n,k,s,basis_kind) for s in grids};reg=np.sqrt(regularization/n)*np.diff(basis_matrix(n,k,1,basis_kind),axis=0);cache={};diagnostics={'rejected':0}
    n_residuals=4*n-1+(2*n if quartic_weight else 0)
    def objective(theta):
        theta=np.asarray(theta,dtype=float)
        if 'theta' in cache and np.array_equal(theta,cache['theta']):return cache['r'],cache['j']
        try:
            with np.errstate(over='raise',invalid='raise',divide='raise'):
                errors=[];jacs=[]
                for s in grids:
                    x,j,_=simulate_tangent(theta,yn[0],bs[s],s)
                    errors.append((x[::s,1]-yn)/np.sqrt(n));jacs.append(j[::s,1,:]/np.sqrt(n))
                jr=np.zeros((n-1,len(theta)));jr[:,5:]=reg
                r=np.r_[errors[0]/np.sqrt(2),errors[1]/np.sqrt(2),errors[0]-errors[1],reg@theta[5:]]
                jac=np.vstack([jacs[0]/np.sqrt(2),jacs[1]/np.sqrt(2),jacs[0]-jacs[1],jr])
                if quartic_weight:
                    weight=np.sqrt(quartic_weight*n/2)
                    r=np.r_[r,weight*errors[0]**2,weight*errors[1]**2]
                    jac=np.vstack([jac,2*weight*errors[0][:,None]*jacs[0],2*weight*errors[1][:,None]*jacs[1]])
                if not np.all(np.isfinite(r)) or not np.all(np.isfinite(jac)):raise NumericalFailure('Nonfinite objective')
            cache.update(theta=theta.copy(),r=r,j=jac);return r,jac
        except (NumericalFailure,FloatingPointError,np.linalg.LinAlgError):
            diagnostics['rejected']+=1
            return np.full(n_residuals,1e6),np.zeros((n_residuals,len(theta)))
    return objective,mu,sd,diagnostics

def fit_model(y,kind='variable',k=8,grids=(2,4),budget=1000,seeds=tuple(range(1,101)),warm=None,regularization=1.,normalization=None,fixed=None,basis_kind='bspline',quartic_weight=0.):
    if kind not in ['integer','constant','variable']:raise ValueError('Invalid model kind')
    if kind!='variable':k=1
    y=np.asarray(y,dtype=float);fn,mu,sd,diag=make_objective(y,k,grids,regularization,normalization,basis_kind,quartic_weight);lo,hi=parameter_bounds(k)
    fixed={} if fixed is None else dict(fixed)
    if kind=='integer':fixed[5]=1.0;hi[5]=1.0
    if any(j not in range(5+k) or not np.isfinite(v) or not lo[j]<=v<=hi[j] for j,v in fixed.items()):raise ValueError('Invalid fixed parameter')
    free=np.array([j for j in range(5+k) if j not in fixed]);starts=[]
    if warm is not None:starts.append(('warm',np.asarray(warm,dtype=float)))
    for seed in seeds:
     rng=np.random.default_rng(seed)

     # richer initial guesses for a,b,q
     a0 = rng.uniform(-10,10)
     b0 = rng.uniform(-10,10)

     q0 = rng.uniform(0.1,0.9)

     q_basis0 = rng.uniform(0.2,0.95,k)

     starts.append(
        (
            str(seed),
            np.r_[
                a0,
                b0,
                rng.uniform(-.8,.8),
                rng.uniform(-.8,.8),
                np.log(q0),
                q_basis0
            ]
        )
    )
    records=[]
    if not starts or not len(free):raise ValueError('Require at least one start and one free parameter')
    for label,initial in starts:
        if initial.shape!=(5+k,) or not np.all(np.isfinite(initial)):raise ValueError('Invalid starting parameter vector')
        initial=initial.copy()
        for j,value in fixed.items():initial[j]=value
        initial[free]=np.clip(initial[free],lo[free]+1e-10,hi[free]-1e-10)
        def unpack(v):
            th=initial.copy();th[free]=v;return th
        r0,_=fn(initial)
        if np.max(abs(r0))>=1e6:
            records.append(dict(label=label,success=False,valid=False,initial=initial.tolist(),message='Invalid initial trajectory'));continue
        opt=least_squares(lambda v:fn(unpack(v))[0],initial[free],jac=lambda v:fn(unpack(v))[1][:,free],bounds=(lo[free],hi[free]),max_nfev=budget,ftol=1e-6,xtol=1e-7,gtol=1e-7,x_scale='jac')
        th=unpack(opt.x);r,_=fn(th);valid=np.all(np.isfinite(r)) and np.max(abs(r))<1e6
        records.append(dict(label=label,success=bool(opt.success),valid=bool(valid),initial=initial.tolist(),theta=th.tolist(),cost=float(opt.cost),optimality=float(opt.optimality),nfev=opt.nfev,message=str(opt.message)))
    eligible=[r for r in records if r['valid']]
    if not eligible:raise NumericalFailure('No valid fitted trajectory')
    # Convergence flags are diagnostics, not substitutes for the objective.
    best=min(eligible,key=lambda r:r['cost'])
    return dict(version=VERSION,kind=kind,k=k,basis_kind=basis_kind,quartic_weight=quartic_weight,n_train=len(y),mu=mu,sd=sd,grids=list(grids),budget=budget,regularization=regularization,theta=best['theta'],selected_success=best['success'],selected_message=best['message'],optimality=best['optimality'],selected_cost=best['cost'],free_indices=free.tolist(),n_parameters=len(free),attempts=records,rejected_evaluations=diag['rejected'],initial_y=float((y[0]-mu)/sd),train_hash=fingerprint(y.tolist()))

def trajectory(result,n_total,substeps,with_jac=False):
    basis=extended_basis(result['n_train'],n_total,result['k'],substeps,result.get('basis_kind','bspline'))
    x,j,res=simulate_tangent(np.array(result['theta']),result['initial_y'],basis,substeps,with_jac)
    return x[::substeps,1]*result['sd']+result['mu'],j[::substeps,1,:]*result['sd'],res

def evaluate_mesh(result,n_total,max_substeps=64):
    if max_substeps<2*max(result['grids']):raise ValueError('Independent mesh must be finer than fitted grids')
    sub=max(result['grids']);previous,_,res=trajectory(result,n_total,sub);history=[];last=None
    while 2*sub<=max_substeps:
        reported=previous
        fine,_,res2=trajectory(result,n_total,2*sub);delta=float(np.sqrt(np.mean((previous-fine)**2)))
        # Verify separately on the held-out forecast horizon and training interval.
        nt=result['n_train'];train=float(np.sqrt(np.mean((previous[:nt]-fine[:nt])**2)))
        future=float(np.sqrt(np.mean((previous[nt:]-fine[nt:])**2))) if nt<n_total else 0.
        passed=max(train,future)/result['sd']<.02
        last=dict(reported_substeps=sub,verification_substeps=2*sub,grid_rmse=delta,train_grid_rmse=train,forecast_grid_rmse=future,relative_grid_rmse=max(train,future)/result['sd'],grid_pass=passed,residual_max=max(res,res2))
        history.append(last.copy())
        if passed:break
        sub*=2;previous=fine;res=res2
    return reported,fine,{**last,'mesh_history':history}

def summary_errors(truth,pred):
    truth=np.asarray(truth,dtype=float);pred=np.asarray(pred,dtype=float)
    if truth.shape!=pred.shape or not len(truth) or not np.all(np.isfinite(pred)):raise ValueError('Invalid metric arrays')
    return {**metrics(truth,pred),'r2':float(1-np.mean((truth-pred)**2)/np.var(truth)) if np.var(truth)>0 else None,'max_abs_error':float(np.max(abs(truth-pred)))}

In [ ]:
DATA = {
'series1': [
    17.216430221049166,
    14.579676674364903,
    11.94292312768064,
    20.581216320246334,
    15.072170900692873,
    8.72748267898388,
    4.565242494226313,
    7.1928406466512635,
    9.820438799076214,
    8.96898713295943,
    5.160343121082169,
    2.5411855273287025,
    5.503464203233248,
    7.801000769822949,
    6.291377983063901,
    7.14780600461895,
    12.785796766743662,
    15.028868360277144,
    16.411085450346427,
    8.036951501154762,
    3.9357967667436435,
    6.46512702078522,
    8.994457274826795,
    11.52378752886837,
    14.053117782909947,
    16.582448036951522,
    19.111778290993097,
    18.04464973056197,
    17.114087759815245,
    17.524249422632803,
    15.002309468822176,
    16.836027713625864,
    15.11200923787529,
    16.051963048498845,
    14.2147806004619,
    11.993071593533479,
    12.324375393659484,
    15.074322905731695,
    17.824270417803906,
    20.57421792987612,
    16.94168591224023,
    13.306004618937653,
    12.98944242824151,
    13.062685582316071,
    12.344110854503464,
    13.115473441108541,
    12.218244803695148,
    15.009237875288708,
    12.803002309468818,
    11.726327944572741,
    10.439030023094672,
    8.712933025404146,
    7.739414934565054,
    7.198229407236344,
    9.714203233256361,
    12.258660508083098,
    9.182448036951467,
    10.455991788555322,
    12.183987682832967,
    13.911983577110613,
    12.97947138824737,
    11.631254811393383,
    10.283038234539395,
    9.20092378752887,
    8.175519630484983,
    13.315819861431915,
    15.66050808314087,
    12.584295612009239,
    12.405080831408785,
    13.035668462920185,
    11.991275340005124,
    10.946882217090064,
    9.989442428241517,
    12.162322665786878,
    14.335202903332238,
    13.598152424942253,
    13.224018475750592,
    13.850654349499631,
    10.038337182448045,
    10.824480369515017,
    7.4341801385680535,
    7.399538106235577,
    7.912240184757515,
    6.341801385681269,
    7.161452865840876,
    8.544194835187934,
    9.926936804534993,
    10.874826789838307,
    9.575981524249395,
    9.247113163972308,
    9.999076212471152,
    8.661431870669741,
    7.219399538106209,
    10.295612009237894,
    11.224018475750558,
    10.224595842956136,
    10.836027713625741,
    7.177829099307176,
    7.861431870669755,
    7.2748267898383405,
    9.077367205542739,
    12.40993071593537,
    15.742494226328002,
    17.58614318706695,
    15.672055427251726,
    18.605080831408785,
    16.179368745188576,
    17.321785989222512,
    18.91685912240189,
    15.057736720554185,
    13.190531177829111,
    13.78868360277137,
    11.011547344110735,
    9.431870669746077,
    12.678983833718338,
    15.926096997690594,
    19.17321016166285,
    21.434180138568113,
    15.281755196304934,
    20.75057736720566,
    23.791570438799067,
    23.065242494226315,
    22.338914549653563,
    21.742989112504112,
    21.205872649290647,
    21.01715605410758,
    21.554272517321028,
    22.09138898053448,
    19.81755196304848,
    19.337182448036973,
    20.089145496535814,
    18.256351039260995,
    19.96535796766746,
    19.159353348729777,
    18.04849884526556,
    19.050808314087767,
    18.93687451886067,
    20.50346420323326,
    19.975057736720547,
    19.006928406466542,
    19.9688221709007,
    19.41339491916862,
    19.530715935334857,
    18.23187066974595,
    20.75635103926102,
    21.303117782909894,
    16.936654569449036,
    17.42494226327945,
    17.91322995710987,
    13.856812933025502,
    17.10392609699769,
    15.574287913779822,
    13.979214780600433,
    15.160739030023105,
    15.591224018475778,
    13.67713625866054,
    14.26558891454965,
    15.55889145496534,
    13.166281755196291,
    16.890877598152457,
    18.7378752886836,
    18.13972286374133,
    20.275981524249456,
    17.703618167821325,
    17.498845265588937,
    17.969976905311757,
    16.952886836027748,
    18.35427251732107,
    18.75692840646649,
    17.90704387990764,
    18.88972286374138,
    18.37297921478058,
    17.262124711316392,
    17.41154734411086,
    18.129330254041577,
    16.8760585065435,
    18.207852193995382,
    18.61478060046191,
    16.751963048498848,
    14.889145496535788,
    13.026327944572726,
    14.975750577367261,
    17.26154734411093,
    18.714395688991484,
    17.147806004618918,
    16.123325635103935,
    17.422170900692848,
    17.544649730561968,
    16.490762124711303,
    16.918783679753723,
    18.998075442648283,
    19.140300230946803,
    17.01327944572747,
    16.201501154734384,
    15.394259320356298,
    14.588584625536116
] , 'series2': [
    6.0313836727835675,
    6.159836575475948,
    5.235819758106939,
    7.925647019384041,
    4.408454328354102,
    9.867220079745325,
    16.69532981292835,
    21.94631267839096,
    21.601157492925406,
    21.256002307459855,
    17.97369787942582,
    18.00918820631041,
    18.044678533195,
    17.43015289431186,
    13.773432767815374,
    10.116712641318886,
    6.459992514822398,
    2.8032723883259116,
    3.2148049544721866,
    4.834310448977691,
    5.747837746623192,
    4.37232621690576,
    2.9968146871883294,
    2.2420515967188077,
    4.475107152274383,
    4.561767784269126,
    3.8910633734725493,
    3.2203589626759723,
    4.054718719693962,
    4.003209117876352,
    0.05685897709190488,
    1.4748334296466368,
    2.8928078822013688,
    1.4103577949479358,
    0.25353570622593746,
    0.7516272158907265,
    1.2497187255555156,
    1.7478102352203044,
    0.8799800336303665,
    1.1700303604655122,
    3.3632481779153958,
    7.027920542587747,
    5.729248555479249,
    4.359391474491808,
    6.0658158969852005,
    6.090592226901253,
    4.8406307183037365,
    4.8641967476501105,
    4.887762776996485,
    7.973498997019267,
    6.968091234263053,
    5.962683471506839,
    7.696609922357467,
    7.540714050641958,
    6.863041591452915,
    7.810733767261481,
    7.148352292663147,
    7.813741061561643,
    5.077596241199641,
    6.305474037162539,
    6.4127862720287645,
    4.245488421465813,
    4.359433115184473,
    8.105795454699436,
    6.760983775651024,
    5.368516243183484,
    3.9760487107159426,
    4.543553098220328,
    5.267607152274374,
    5.99166120632842,
    4.673214689636333,
    3.9572333204986863,
    4.028528380578791,
    3.7295968163571023,
    3.1124315715072317,
    3.8071371360669213,
    4.055102711777053,
    4.0072341504981885,
    4.237877422544647,
    4.96193147659871,
    3.776088657790293,
    3.2703046999583014,
    4.270186581461238,
    4.789464987492471,
    4.09426152706486,
    4.399457638362208,
    4.704653749659556,
    5.275002585238397,
    6.207853591342851,
    7.140704597447305,
    8.07355560355176,
    8.960545540900032,
    7.75364980630287,
    6.546754071705708,
    5.339858337108547,
    4.132962602511385,
    5.064146139962461,
    5.4080351678386265,
    4.885965930900339,
    4.9879888316636976,
    5.090011732427056,
    4.445088711810756,
    3.913930022229537,
    4.39437845272281,
    4.874826883216084,
    4.929340485607721,
    4.929073818941055,
    4.928807152274389,
    4.734221735607709,
    3.774727363635028,
    3.278822475920367,
    3.0447353222538043,
    4.550698310601989,
    4.273638483867105,
    3.1537951418305625,
    4.847373980255041,
    4.304743095961154,
    4.411103392875857,
    6.779512717198604,
    4.0224300865071765,
    4.905181157037882,
    6.017396196166832,
    7.092774166888215,
    5.945540346428902,
    5.261761287612725,
    8.17005396078502,
    8.027234811848853,
    7.884415662912687,
    9.6260873743423,
    10.59612334872166,
    9.503699106297402,
    8.624958930930525,
    7.989122303789553,
    7.996112822721351,
    8.03160314960594,
    8.06709347649053,
    6.431859654696161,
    4.752815418338395,
    3.0737711819806295,
    4.013045748765569,
    5.266574013872792,
    5.545315377830554,
    3.936779423355653,
    3.7003134229014014,
    5.380566448203949,
    5.7609326516868515,
    5.304029008913647,
    4.847125366140442,
    4.390221723367237,
    3.9333180805940313,
    3.476414437820826,
    3.019510795047621,
    3.304005001736732,
    3.73599424904856,
    4.035212261374976,
    3.9208429159784877,
    5.543444648987436,
    10.081467262713424,
    14.619489876439413,
    19.157512490165402,
    20.870462778601357,
    20.614146430406027,
    20.357830082210697,
    20.10151373401537,
    17.1601973087725,
    13.707373539210241,
    10.25454976964798,
    6.80172600008572,
    4.696737378127349,
    4.219971400232107,
    3.7432054223368656,
    3.266439444441624,
    2.8244024606361045,
    2.45909788250519,
    2.0937933043742762,
    1.9854622311808114,
    1.9971092638547387,
    2.008756296528666,
    2.0204033292025936,
    1.9568124502876483,
    1.6015806622081776,
    1.246348874128707,
    2.1714350454492966,
    4.740424053682849,
    4.221145884668772,
    4.235963316657894,
    5.704182494740071,
    3.9138079078059276,
    4.5499115235641625,
    4.5350818997492155,
    3.9844932732708256,
    4.031913202096447,
    3.0291092981971897,
    2.155102658565567,
    2.7489612565284167,
    3.3428198544912666,
    3.8920408140271103,
    3.8957447920995603,
    3.89944877017201,
    3.90315274824446,
    3.9068567263169096,
    3.9105607043893595,
    3.9142646824618095,
    3.9179686605342594,
    3.9216726386067093,
    3.925376616679159,
    3.9290805947516088,
    3.9327845728240587,
    3.9364885508965086,
    3.9401925289689586,
    4.838154347896781,
    6.231040804805069,
    5.747826746869224,
    4.014293606672611,
    4.650397222430857,
    4.788730005642974,
    4.292825117928312
] , 'series3': [
    0.5392048964563531,
    1.0437072349586882,
    0.9762136233743618,
    0.9087200117900354,
    4.76100189143363,
    5.747593116829605,
    3.232007673791638,
    0.7164222307536718,
    1.1769222230571128,
    3.03009409172351,
    4.883265960389908,
    3.5224710104753765,
    2.6138343323180298,
    8.608337280928009,
    6.60094495588479,
    4.703288869424243,
    5.820545844200986,
    4.676715470655363,
    3.4666401741078365,
    2.153073686790386,
    2.7535489397728434,
    3.5000132701858604,
    4.246477600598878,
    4.992941931011895,
    5.739406261424913,
    5.2941224737681924,
    4.150299126798415,
    6.064817540396133,
    2.1046273162522056,
    2.8022150355504536,
    1.8989550501929333,
    2.947830764060922,
    2.195478999030589,
    1.4264512212528107,
    0.7957255976700432,
    0.16499997408727585,
    0.33385338067725606,
    0.7631713394603755,
    0.7246621179303219,
    0.14880161068394182,
    0.19915797822216716,
    0.6573952912890941,
    1.1156326043560212,
    1.573869917422948,
    1.9802138115602665,
    0.8582180323663392,
    0.9998995270282622,
    0.8399833611984127,
    0.23975677509569737,
    0.719297612682938,
    0.5987563208288728,
    0.0228958135824886,
    0.4973300224311508,
    1.0803945385601823,
    1.0856384378060957,
    1.08586165209181,
    1.0860848663775244,
    1.0863080806632388,
    1.0865312949489532,
    1.0867545092346673,
    1.0869777235203817,
    1.087200937806096,
    2.350863370501758,
    6.399379809956433,
    10.447896249411107,
    14.496412688865782,
    17.41748904531717,
    11.270839560781113,
    8.40861777581789,
    9.155082106230907,
    9.901546436643924,
    9.184864974350955,
    8.066061424779347,
    5.848574049826254,
    4.078160759234658,
    4.8733795092346615,
    5.668598259234666,
    5.158160759234659,
    3.6954241689700127,
    3.049431730406686,
    2.4034392918433594,
    1.7381596947002211,
    1.0462974691135467,
    0.3544352435268723,
    0.23002035430508697,
    0.790099579657203,
    1.0481384378060978,
    1.0483616520918126,
    1.0485848663775272,
    1.0488080806632418,
    2.6644235631986906,
    6.822351611166693,
    8.030665879641818,
    7.991415631379652,
    12.408837842568047,
    12.52017842227814,
    12.275036810715216,
    14.390242966277285,
    11.2076962099944,
    8.025149453711519,
    4.842602697428637,
    3.2096468183322338,
    3.921502865098767,
    4.6333589118653,
    3.5932540932612307,
    2.5573870758206376,
    3.5008253324792484,
    1.9503858800606688,
    2.5793401858603158,
    3.104562298179395,
    2.585441695164317,
    2.066321092149239,
    5.432527469555687,
    9.455576956849514,
    10.491752004224594,
    5.6327840683529145,
    6.788163660428324,
    11.111306896502672,
    10.857889377655615,
    6.673925561866197,
    7.645758746522812,
    9.667157051607546,
    11.68855535669228,
    13.709953661777014,
    15.127896595565602,
    16.443580048803035,
    14.524163794683002,
    9.539022373023187,
    4.553880951363372,
    2.691925372544032,
    4.0076088257814355,
    5.32329227901884,
    5.877777236507397,
    5.155277236507403,
    4.432777236507409,
    3.747766413996577,
    3.1170407904137845,
    2.4016408728710035,
    1.6791408728710018,
    0.9566408728710001,
    2.810755570055104,
    3.348981640211737,
    2.394513129924374,
    3.765418302338156,
    2.66727458222733,
    2.578290085226943,
    2.6873140691739534,
    1.922832835414952,
    3.0176033056238305,
    2.7233966660974342,
    3.676228910106131,
    2.4711195252090734,
    2.6405922285329195,
    3.1166652235203824,
    3.116888437806097,
    3.1171116520918116,
    3.1173348663775258,
    4.091290649324082,
    4.032274255881463,
    5.627405623440823,
    7.262164862770029,
    7.663900973881139,
    8.065637084992249,
    6.806923638317049,
    5.014388646093092,
    3.221853653869135,
    4.670711515604121,
    6.0595619839219825,
    6.036110524720077,
    6.243444650079733,
    6.803523875431856,
    5.987211282853208,
    6.058289714225756,
    3.5318724337625023,
    3.026412081617745,
    2.477706767961532,
    1.0903607720683062,
    0.22379299484169504,
    0.9702573252547047,
    1.7167216556677143,
    2.0403742362818145,
    2.020870514197447,
    2.7578862894850484,
    3.8281494367708913,
    3.2522889295245245,
    4.12494480774213,
    4.0065119719212365,
    5.873500061966594,
    8.426517606307064,
    7.018043030035852,
    6.08385714081363,
    6.469025561866255,
    6.854193982918879,
    5.552954870245502,
    3.1136610976823254,
    1.3521128874053003,
    5.171492211002546,
    8.990871534599792,
    11.201699435803382,
    6.372510517245274,
    7.5058645003733035,
    5.832969063953282,
    4.160073627533261,
    5.136754509234668,
    5.136977723520382,
    4.951693533624868,
    3.982409997039495,
    3.0131264604541217,
    6.256663600143848,
    6.004578583308736,
    5.876086701015514,
    9.034511358549947,
    8.004200642943708,
    4.722625934929365,
    1.4410512269150217,
    3.392987214180936,
    6.18435698242723,
    8.975726750673523
] , 'series4': [
    15.713688719002512,
    8.162192492096441,
    6.987511253480729,
    7.035780834537195,
    4.777711617779481,
    1.1056533299834548,
    2.190316353662871,
    3.2749793773422877,
    3.6623746414588876,
    1.9128300148650605,
    2.4365094319871003,
    2.96018884910914,
    2.0346879383623375,
    2.0157444046646966,
    1.91647300211461,
    1.2378104988619354,
    5.204815442916066,
    8.858003056759419,
    3.7243054247011615,
    2.4135586123149784,
    1.599298919366762,
    0.7850392264185455,
    -0.02922046652967092,
    3.2743964993823664,
    3.8134857525699952,
    2.3689684483805555,
    5.873303603207509,
    5.018294495739383,
    3.035260407787278,
    1.0256845555884877,
    3.573485752569975,
    2.853215130660019,
    1.3597985290466932,
    1.7953436760672388,
    1.5815003245189896,
    0.7793145321692634,
    0.8718464083075523,
    7.704487574063606,
    8.79242928626763,
    5.256108703389697,
    4.3637225467411955,
    5.054099856134903,
    5.5981539805168925,
    5.436509431987137,
    3.4773771253138017,
    5.051280089380705,
    6.625183053447608,
    7.71301216422754,
    6.033959340912424,
    5.71629085275213,
    7.238330925611871,
    8.760370998471613,
    5.6365094319871085,
    8.143248958398766,
    8.714469359127358,
    8.022301781713905,
    6.232866444737555,
    6.758913803571788,
    4.955635115047231,
    9.02193748298893,
    8.479340553506757,
    7.495525825429806,
    3.7039046961036806,
    3.1341414902749065,
    5.948349140548124,
    5.917383748926993,
    6.31359504218747,
    7.106454787178394,
    6.12919743615049,
    7.759902614396668,
    8.822301781713882,
    6.956051456447175,
    8.07059738410785,
    9.185143311768527,
    11.08423256495614,
    10.585780834537196,
    10.087329104118252,
    13.883868266231188,
    13.250717082260332,
    10.134870087724849,
    6.9869648053932965,
    7.0827753700563205,
    5.012902874610055,
    5.089697045830455,
    3.026309067688336,
    3.8620103427339525,
    2.771508130920239,
    2.228911201438059,
    2.6824110713313396,
    4.36674622615831,
    5.342064987542649,
    5.991263530347763,
    5.193303603207512,
    7.702447501203878,
    7.042338211586361,
    10.624487574063606,
    13.377857337269438,
    13.699460251659232,
    11.247074095010776,
    8.794687938362319,
    6.782957519418792,
    5.927948411950664,
    5.425164128838521,
    5.208562515515601,
    4.424487574063641,
    4.017201599564518,
    4.017930197014426,
    4.129405606850454,
    5.043066809036286,
    4.980407428344119,
    5.959642401021728,
    6.5786379773943375,
    7.095525825429721,
    3.4328664447375936,
    1.7221196323514274,
    0.24889558863552647,
    3.7042689948285887,
    5.12976990557549,
    2.2780394866319345,
    4.615744404664686,
    6.451299960220263,
    2.1543236396374006,
    3.331647445157883,
    6.594418917765416,
    9.857190390372947,
    13.11996186298048,
    6.917310889181978,
    4.122848229801313,
    4.186964805393299,
    2.435999413772171,
    1.4973142009885805,
    2.4142872097648684,
    3.3312602185411566,
    3.4838682662312164,
    1.9898063354479418,
    2.066600506668344,
    2.7453618910035056,
    3.9731214538450392,
    5.717019450202043,
    7.460917446559047,
    9.169634594620456,
    10.148349140548126,
    10.931591399200183,
    6.241609614136412,
    3.2670376651382984,
    4.104487574063617,
    3.338877373699311,
    4.992064987542618,
    4.594979377342331,
    4.262593220693817,
    7.077748047651916,
    8.003722546741203,
    6.128677009400576,
    7.79261143563012,
    6.932028557670167,
    6.781500324518978,
    8.037966626886913,
    7.563285388271254,
    7.088604149655594
] , 'series5': [
    15.414807162534435,
    13.438085399449033,
    11.461363636363629,
    9.484641873278226,
    12.16829004329007,
    16.986688311688333,
    18.180757575757553,
    13.010151515151495,
    7.839545454545437,
    3.037878787878798,
    6.205303030303035,
    6.284090909090904,
    5.1426767676767575,
    12.284090909090953,
    8.931818181818144,
    3.7136363636363616,
    3.1999999999999993,
    5.595129870129884,
    12.150108225108252,
    18.003787878787872,
    16.63409090909091,
    16.045170454545453,
    16.494602272727274,
    16.944034090909092,
    15.034722222222213,
    12.438005050505033,
    16.028030303030302,
    15.91388888888889,
    23.01450216450217,
    23.03896103896104,
    23.063419913419914,
    18.129090909090916,
    19.361818181818187,
    20.59454545454546,
    21.82727272727273,
    20.467272727272732,
    18.66508838383839,
    17.45233585858586,
    16.239583333333336,
    15.02683080808081,
    13.443939393939397,
    12.275000000000007,
    14.13075757575755,
    10.398333333333293,
    18.971590909090907,
    18.372348484848487,
    18.965909090909037,
    22.825909090909107,
    21.952727272727287,
    21.079545454545467,
    20.206363636363644,
    14.50252525252531,
    14.709469696969695,
    13.979999999999992,
    11.993939393939378,
    15.446590909090894,
    14.135606060606104,
    9.056313131313171,
    7.59772727272727,
    8.625000000000005,
    7.424242424242417,
    4.7490530303030525,
    2.0738636363636873,
    3.012626262626264,
    2.2130681818181808,
    6.468181818181834,
    13.830303030303073,
    15.026767676767676,
    15.996969696969693,
    20.022727272727252,
    17.825505050505058,
    21.988636363636413,
    16.534343434343413,
    18.645959595959575,
    18.551136363636406,
    14.014015151515189,
    9.476893939393971,
    4.939772727272755,
    2.4588744588744507,
    3.8041125541125442,
    5.095959595959601,
    4.430134680134684,
    3.7643097643097683,
    3.10539772727273,
    2.6131628787878833,
    2.1209280303030367,
    2.7573863636363427,
    5.036026936026935,
    4.6745791245791235,
    4.313131313131312,
    4.072510822510825,
    5.417748917748927,
    6.762987012987029,
    2.0189393939394007,
    3.7310606060605904,
    2.269318181818198,
    4.844949494949538,
    9.890909090909084,
    9.37727272727272,
    9.791666666666721,
    15.123295454545426,
    17.095129870129867,
    17.510930735930735,
    17.926731601731607,
    15.193181818181777,
    11.654797979797936,
    16.571753246753275,
    15.691233766233783,
    15.423989898989873,
    18.27525252525252,
    19.24545454545454,
    19.37899449035815,
    15.238774104683243,
    11.098553719008335,
    6.958333333333426,
    5.847077922077886,
    7.6081168831168675,
    8.716477272727275,
    7.047159090909115,
    10.215909090909014,
    9.71712121212131,
    5.368333333333452,
    15.306818181818175,
    15.94886363636363,
    14.962121212121227,
    15.705113636363615,
    15.54305555555559,
    14.487247474747514,
    7.046590909091202,
    9.442234848484873,
    10.732196969696961,
    10.132954545454535,
    12.707575757575697,
    11.524431818181801,
    11.670454545454577,
    13.040151515151548,
    14.409848484848519,
    17.096818181818225,
    20.623787878787844,
    21.330627705627695,
    21.893181818181805,
    21.118722943722968,
    19.553354978354978,
    17.09520202020196,
    12.605639730639666,
    8.116077441077373,
    5.438636363636372,
    6.637121212121194,
    6.977727272727273,
    6.875,
    16.423863636363617,
    16.710151515151537,
    16.059545454545468,
    17.952272727272707,
    20.848958333333304,
    15.56278409090902,
    10.276609848484734,
    8.912500000000032,
    14.08579545454546,
    13.957386363636363,
    18.80075757575779,
    16.071969696969685,
    16.642676767676758,
    12.893939393939359,
    13.749999999999991,
    16.643333333333356,
    19.93238636363635,
    19.03352272727272,
    19.745909090909226,
    16.04772727272736,
    15.593181818181801,
    19.05606060606059,
    18.114393939393935,
    18.820454545454552,
    17.79829545454546,
    17.965340909090866,
    19.819090909090985,
    17.35363636363645,
    14.888181818181916,
    12.422727272727382,
    13.765814393939324,
    16.05577651515149,
    17.569696969696963,
    14.544949494949527,
    11.988636363636363,
    11.988636363636363,
    11.422979797979783,
    11.380176767676769,
    12.435984848484852
] , 'series6': [
    8.671874999999993,
    9.927083333333327,
    9.438125000000031,
    5.420357142857149,
    7.50758928571429,
    9.59482142857143,
    7.228281249999986,
    4.424375000000021,
    8.077031250000005,
    8.916666666666666,
    7.072968750000002,
    7.713125000000002,
    8.35328125,
    8.9934375,
    6.043750000000004,
    7.248749999999997,
    8.96906249999999,
    10.03125,
    10.03125,
    10.03125,
    10.03125,
    10.03125,
    9.598562499999995,
    7.173500000000002,
    5.252604166666657,
    7.687708333333326,
    9.082421875000005,
    9.931249999999995,
    9.272265625,
    8.31875,
    7.113749999999996,
    7.741666666666663,
    7.042499999999993,
    5.512767857142841,
    4.103348214285703,
    2.6939285714285663,
    4.327031250000028,
    9.184687500000067,
    8.799937499999983,
    7.308749999999987,
    9.64737499999999,
    8.412249999999993,
    7.177124999999997,
    5.942000000000001,
    7.6937499999999455,
    9.539624999999983,
    7.566437499999957,
    8.106250000000037,
    9.73802083333336,
    10.324999999999955,
    6.920874999999966,
    4.618750000000033,
    9.514062500000051,
    7.35406249999998,
    5.550000000000013,
    7.960000000000042,
    7.60859374999995,
    5.160937499999962,
    10.99758928571429,
    11.36339285714286,
    11.729196428571433,
    11.53906249999996,
    8.714843749999957,
    5.890624999999957,
    4.278124999999985,
    3.2237499999999866,
    2.169374999999988,
    6.382916666666797,
    8.796093749999986,
    8.356770833333325,
    8.061250000000033,
    9.307916666666674,
    9.697031250000006,
    9.170624999999957,
    5.421093750000014,
    6.8746250000000195,
    8.328156250000024,
    9.781687500000029,
    10.414999999999909,
    6.799999999999897,
    5.765703125000043,
    7.290781250000034,
    8.815859375000027,
    9.16187499999993,
    6.548697916666696,
    8.193020833333367,
    9.837343750000041,
    6.260535714285741,
    6.981383928571451,
    7.702232142857161,
    8.024427083333332,
    8.011875,
    8.044687500000025,
    8.873125000000025,
    9.701562500000023,
    9.35062499999997,
    8.089140624999976,
    8.922500000000026,
    9.976875000000035,
    11.031249999999941,
    10.05281250000003,
    10.730625000000025,
    9.198281249999857,
    8.499821428571456,
    9.188392857142883,
    9.87696428571431,
    8.511874999999902,
    6.5537499999998685,
    7.698359375000139,
    9.986718749999936,
    8.16039062499996,
    7.246875000000011,
    7.811718750000022,
    10.633125000000181,
    9.945052083333286,
    8.727499999999969,
    8.400125000000017,
    9.39425000000002,
    9.109562499999914,
    6.202499999999865,
    4.767062500000166,
    7.674125000000136,
    8.960624999999759,
    4.734062499999974,
    4.150390624999972,
    3.5667187499999704,
    3.5621875000010315,
    8.367249999999789,
    4.029249999999834,
    3.168750000000075,
    5.277500000000082,
    7.386250000000088,
    8.58406249999991,
    6.5506249999999255,
    4.517187499999942,
    5.453750000000056,
    7.487187500000063,
    8.741249999999956,
    7.520781249999951,
    6.547916666666633,
    5.769687499999963,
    5.032053571428555,
    4.69852678571427,
    4.364999999999984,
    4.358125000000067,
    5.774000000000081,
    6.572812499999738,
    2.480833333333198,
    2.8705357142857535,
    3.9464285714286196,
    5.022321428571486,
    7.189583333333401,
    8.382031250000072,
    9.574479166666743,
    9.741666666666642,
    9.339999999999979,
    9.063593750000019,
    9.47781250000002,
    9.892031250000022,
    6.806249999999723,
    3.9899999999998683,
    1.5799999999998757,
    2.3867968750000976,
    4.51375000000027,
    9.358854166666898,
    7.06187500000006,
    8.291979166666717,
    9.522083333333374,
    7.473437499999752,
    5.225000000000286,
    9.442500000000155,
    7.912499999999861,
    7.5908854166667,
    8.406770833333379,
    9.101562500000027,
    9.52833333333335,
    9.955104166666676,
    6.8384999999998515,
    2.9523749999997584,
    2.559375000000043,
    3.3501562500000275,
    5.196875000000147,
    8.364791666666708,
    9.002343749999987,
    8.672851562499988,
    8.34335937499999,
    8.01386718749999,
    9.377083333333363,
    9.615859374999989,
    8.994531249999987,
    9.582343750000026,
    9.59249999999992,
    8.123906249999925,
    6.655312499999929,
    5.186718749999934,
    4.792500000000254,
    10.516250000000337,
    7.560874999999727,
    3.689812499999687,
    8.385416666667155,
    9.299999999999745,
    6.2874999999996835,
    5.667604166666629,
    5.2408333333333035,
    5.643125000000076,
    7.11925000000005,
    6.295312499999857,
    2.2522500000001022,
    4.692375000000094,
    6.993749999999967,
    6.089999999999952,
    6.461250000000048,
    6.722187499999782,
    4.2368749999998006,
    5.054947916666764,
    6.272500000000102,
    7.166718750000034,
    7.580937500000031,
    7.995156250000028,
    5.9624999999999995,
    5.949947916666665,
    5.968437500000132,
    8.812890624999916,
    7.589062499999912,
    7.234375000000031,
    7.68625000000003,
    8.096328125000024,
    8.454062500000026,
    7.925859374999942,
    7.15390624999994,
    6.381953124999937,
    6.4307500000000815,
    7.424875000000078
]

}

In [ ]:
DATA_FULL = {name: list(values) for name, values in DATA.items()}
N_CALIBRATION_POINTS = 60  # Calibrate on indices 0 .. N_CALIBRATION_POINTS-1.
if (not isinstance(N_CALIBRATION_POINTS, (int, np.integer))
        or isinstance(N_CALIBRATION_POINTS, bool) or N_CALIBRATION_POINTS < 4):
    raise ValueError("N_CALIBRATION_POINTS must be an integer of at least four")
if N_CALIBRATION_POINTS > min(len(values) for values in DATA_FULL.values()):
    raise ValueError("N_CALIBRATION_POINTS exceeds the length of the shortest series")
# Separate this count-based prefix protocol from previous checkpoint directories.
OUT = Path.cwd() / f"combined_online_observer_output_prefix{int(N_CALIBRATION_POINTS)}"
OUT.mkdir(parents=True, exist_ok=True)
DATA = {name: values[:N_CALIBRATION_POINTS] for name, values in DATA_FULL.items()}
FULL_LENGTHS = {name: len(values) for name, values in DATA_FULL.items()}
CALIBRATION_LENGTHS = {name: len(values) for name, values in DATA.items()}
POST_CALIBRATION_LENGTHS = {name: FULL_LENGTHS[name] - CALIBRATION_LENGTHS[name] for name in DATA_FULL}
print("Full lengths:", FULL_LENGTHS)
print("Calibration lengths:", CALIBRATION_LENGTHS)
print("Calibration indices:", (0, N_CALIBRATION_POINTS - 1))
print("Post-calibration samples per series:", POST_CALIBRATION_LENGTHS)


In [ ]:
K_VARIABLE = 48
BASE_DEFAULTS = {name: item.default for name, item in inspect.signature(fit_model).parameters.items()
                 if item.default is not inspect.Parameter.empty}
FIT_GRIDS = tuple(BASE_DEFAULTS["grids"])
FIT_BUDGET = int(BASE_DEFAULTS["budget"])
SEEDS = tuple(BASE_DEFAULTS["seeds"])
REGULARIZATION = float(BASE_DEFAULTS["regularization"])
VERIFY_MAX_SUBSTEPS = int(inspect.signature(evaluate_mesh).parameters["max_substeps"].default)
FIT_OPTIONS = dict(kind="variable", k=K_VARIABLE, grids=FIT_GRIDS, budget=FIT_BUDGET,
                   seeds=SEEDS, warm=None, regularization=REGULARIZATION, normalization=None,
                   fixed=None, basis_kind=BASE_DEFAULTS["basis_kind"],
                   quartic_weight=BASE_DEFAULTS["quartic_weight"])
SERIES_NAMES = tuple(f"series{i}" for i in range(1, 7))
EXPECTED_LENGTHS = {series: N_CALIBRATION_POINTS for series in SERIES_NAMES}

# Requested pre-fit print. Tuple order is (lower_bounds, upper_bounds).
print(parameter_bounds(48))
lower, upper = parameter_bounds(K_VARIABLE)
assert lower.shape == upper.shape == (53,)
assert (lower[0], upper[0]) == (-10.0, 10.0)
assert (lower[1], upper[1]) == (-10.0, 10.0)
assert np.array_equal(lower[2:5], np.array([-3., -3., np.log(.03)]))
assert np.array_equal(upper[2:5], np.array([3., 3., np.log(3.)]))
assert np.all(lower[5:] == .03) and np.all(upper[5:] == .98)
assert tuple(DATA) == tuple(DATA_FULL) == SERIES_NAMES
assert all(DATA[s] == DATA_FULL[s][:N_CALIBRATION_POINTS] for s in SERIES_NAMES)
assert {s:len(DATA[s]) for s in SERIES_NAMES} == EXPECTED_LENGTHS
assert FIT_GRIDS == (2,4) and FIT_BUDGET == 1000 and SEEDS == tuple(range(1,101))
assert REGULARIZATION == 1.0 and FIT_OPTIONS["warm"] is None and FIT_OPTIONS["fixed"] is None
assert hasattr(BSpline, "design_matrix")
for series in SERIES_NAMES:
    y = np.asarray(DATA[series], float)
    assert y.ndim == 1 and np.all(np.isfinite(y)) and y.std() > 1e-12
    for sub in (1, *FIT_GRIDS):
        B = basis_matrix(len(y), K_VARIABLE, sub, FIT_OPTIONS["basis_kind"])
        assert B.shape == ((len(y)-1)*sub+1, K_VARIABLE)
        assert np.min(B) >= -1e-14 and np.max(abs(B.sum(axis=1)-1)) < 1e-12
print("a bounds:", (lower[0],upper[0]), "b bounds:", (lower[1],upper[1]))
print("Fit settings:", FIT_OPTIONS)
print("Free parameters per series:", 5+K_VARIABLE)
print("Calibration data lengths:", EXPECTED_LENGTHS)
print("Full data lengths:", FULL_LENGTHS)
print("PREFIT STRUCTURAL CHECKS PASSED (no fit has run yet)")

In [ ]:
# Fixed provenance from the reviewed attachments; do not derive the expected hash from DATA_FULL.
SOURCE_PROVENANCE = {'source_notebooks': {'combined_prefix_calibration_causal_observer_with_correct_data(2).ipynb': '6c0998368253632b3ab23b63027cd1b908577c53bb26f3c92176f9ac80567780', 'combined_prefix_calibration_causal_observer_phase3(2)(1).ipynb': '05aaef2ca37ba1b7212702dd5b31ac418c858cc16422e8dc55de85bf3f4e9ffa'}, 'data_sha256': 'a0c341ccb0f00577a182033b04747093629e1e99e483ce6c41ea53c7496614b8', 'full_series_lengths': {'series1': 196, 'series2': 217, 'series3': 218, 'series4': 159, 'series5': 189, 'series6': 233}, 'data_origin': 'QIDS, Figure 4(a), user-supplied digitized and linearly interpolated arrays; unchanged', 'delivered_core_source_sha256': 'cfe259cd551365c7ae890a5dcf59e0f7699f6a2ff568093c4319b74a7daf61fd', 'core_source_hash_convention': 'UTF-8 of zero-based code cells 3,4,5 joined by two newline characters', 'unchanged_core_cells': ['input cell 3', 'input cell 4', 'input cell 5'], 'changed_core_functions': [], 'base_fit_defaults': {'grids': [2, 4], 'budget': 1000, 'seeds': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100], 'regularization': 1.0}, 'builder_numerical_execution': False}
PROTOCOL_VERSION = "combined-prefix-count-six-series-causal-observer-v2"
assert set(DATA) == set(DATA_FULL) == set(SERIES_NAMES)
if fingerprint(DATA_FULL) != SOURCE_PROVENANCE["data_sha256"]:
    raise ValueError("Full-data SHA256 mismatch: the six reviewed QIDS series were modified. "
                     "Reload the original DATA cell; do not bypass integrity validation.")
assert FULL_LENGTHS == SOURCE_PROVENANCE["full_series_lengths"]
assert all(CALIBRATION_LENGTHS[s] == N_CALIBRATION_POINTS for s in SERIES_NAMES)

def json_ready(value):
    if isinstance(value, dict): return {str(k):json_ready(v) for k,v in value.items()}
    if isinstance(value, (list,tuple)): return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray): return json_ready(value.tolist())
    if isinstance(value, np.generic): return json_ready(value.item())
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(json_ready(value),ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
    temporary.replace(path)

def code_contract(c):
    # Ignore notebook cell filenames/line numbers so Restart & Run All can resume.
    # Include nested code, constants and parameter defaults to detect real edits.
    constants = [code_contract(v) if isinstance(v, types.CodeType)
                 else dict(type=type(v).__name__, value=repr(v)) for v in c.co_consts]
    return dict(bytecode=c.co_code.hex(), constants=constants, names=c.co_names,
                varnames=c.co_varnames, freevars=c.co_freevars, cellvars=c.co_cellvars,
                argcount=c.co_argcount, posonly=c.co_posonlyargcount,
                kwonly=c.co_kwonlyargcount, flags=c.co_flags,
                exceptiontable=getattr(c,"co_exceptiontable",b"").hex())

CORE_FUNCTIONS = (field,jacobian,l1_kernel,implicit_step,metrics,interpolation_basis,
                  simulate_tangent,basis_matrix,extended_basis,parameter_bounds,
                  make_objective,fit_model,trajectory,evaluate_mesh,summary_errors,fingerprint)
CORE_RUNTIME_HASH = fingerprint({f.__name__:dict(code=code_contract(f.__code__),
                                  defaults=repr(f.__defaults__),kwdefaults=repr(f.__kwdefaults__))
                                 for f in CORE_FUNCTIONS})

def case_spec(series):
    lo,hi = parameter_bounds(K_VARIABLE)
    return json_ready(dict(protocol=PROTOCOL_VERSION, series=series,
        series_names=SERIES_NAMES, n_calibration_points=N_CALIBRATION_POINTS,
        full_data_hash=fingerprint(DATA_FULL[series]),
        data_hash=fingerprint(DATA[series]), source_core_hash=SOURCE_PROVENANCE["delivered_core_source_sha256"],
        runtime_core_hash=CORE_RUNTIME_HASH, environment=ENVIRONMENT, options=FIT_OPTIONS,
        bounds_lower=lo, bounds_upper=hi, verification_max_substeps=VERIFY_MAX_SUBSTEPS))

def selected_attempt(fit, y):
    # Audit the unchanged selection rule: minimum cost among valid attempts.
    assert fit["kind"] == "variable" and fit["k"] == K_VARIABLE
    assert fit["n_train"] == len(y) and fit["train_hash"] == fingerprint(y.tolist())
    assert fit["n_parameters"] == 53 and fit["free_indices"] == list(range(53))
    assert fit["grids"] == list(FIT_GRIDS) and fit["budget"] == FIT_BUDGET
    assert fit["regularization"] == REGULARIZATION
    assert np.isclose(fit["mu"],y.mean(),rtol=0,atol=1e-12)
    assert np.isclose(fit["sd"],y.std(),rtol=0,atol=1e-12)
    assert [a["label"] for a in fit["attempts"]] == [str(seed) for seed in SEEDS]
    lo,hi = parameter_bounds(K_VARIABLE)
    for attempt in fit["attempts"]:
        start = np.asarray(attempt["initial"],float)
        assert start.shape == lo.shape and np.all(np.isfinite(start))
        assert np.all(start >= lo) and np.all(start <= hi)
    valid_attempts = [a for a in fit["attempts"] if a["valid"]]
    assert valid_attempts, "fit_model returned without a valid attempt"
    best = min(valid_attempts,key=lambda a:a["cost"])
    assert np.isfinite(best["cost"]) and np.array_equal(fit["theta"],best["theta"])
    assert fit["selected_cost"] == best["cost"]
    assert bool(fit["selected_success"]) == bool(best["success"])
    theta = np.asarray(fit["theta"],float)
    assert theta.shape == (53,) and np.all(np.isfinite(theta))
    assert np.all(theta >= lo-1e-10) and np.all(theta <= hi+1e-10)
    return best

def save_checkpoint(path,record):
    record = json_ready(record)
    record["fit_hash"] = fingerprint(record["fit"]) if record.get("fit") is not None else None
    evaluation = record.get("evaluation")
    record["evaluation_hash"] = fingerprint(evaluation) if evaluation is not None else None
    atomic_json(path,record)

def load_checkpoint(path,spec,y):
    if not RESUME or not path.exists(): return None
    record = json.loads(path.read_text(encoding="utf-8"))
    if record.get("spec") != spec:
        raise RuntimeError("Checkpoint specification mismatch: " + str(path) +
                           ". Use a new OUT directory for a different experiment; stale parameters were not loaded.")
    if record.get("fit") is not None:
        if fingerprint(record["fit"]) != record.get("fit_hash"):
            raise RuntimeError("Checkpoint fit checksum mismatch: " + str(path))
        selected_attempt(record["fit"],y)
    if record.get("evaluation") is not None:
        if fingerprint(record["evaluation"]) != record.get("evaluation_hash"):
            raise RuntimeError("Checkpoint evaluation checksum mismatch: " + str(path))
    return record

In [ ]:
NUMERICAL_ERRORS = (NumericalFailure, FloatingPointError, np.linalg.LinAlgError, OverflowError)

def evaluate_refit(fit,y):
    assert fit["n_train"] == len(y)  # observed interval only
    sub = max(fit["grids"])
    base,_,base_res = trajectory(fit,len(y),sub,False)
    mesh_error = None
    try:
        reported,verification,mesh = evaluate_mesh(fit,len(y),max_substeps=VERIFY_MAX_SUBSTEPS)
        rmse_grid_source = "evaluate_mesh_reported_grid"
    except NUMERICAL_ERRORS as exc:
        reported,verification = base,None
        mesh_error = type(exc).__name__ + ": " + str(exc)
        mesh = dict(reported_substeps=sub,verification_substeps=None,grid_pass=False,
                    relative_grid_rmse=None,residual_max=float(base_res),mesh_history=[])
        rmse_grid_source = "fitted_grid_fallback_verification_failed"
    theta = np.asarray(fit["theta"],float)
    q = basis_matrix(len(y),fit["k"],1,fit["basis_kind"]) @ theta[5:]
    assert reported.shape == y.shape and np.all(np.isfinite(reported))
    assert np.all((q >= .03-1e-12) & (q <= .98+1e-12))
    free_lower,free_upper = parameter_bounds(fit["k"])
    relative_edge = np.minimum(theta-free_lower,free_upper-theta)/(free_upper-free_lower)
    return json_ready(dict(prediction=reported,verification_prediction=verification,
        fit_grid_prediction=base,q=q,mesh=mesh,mesh_error=mesh_error,rmse_grid_source=rmse_grid_source,
        metrics=summary_errors(y,reported),rmse_excluding_initial=metrics(y[1:],reported[1:])["rmse"],
        rmse_fit_grid=metrics(y,base)["rmse"],initial_y_exact=bool(reported[0] == y[0]),
        near_bound_indices=np.flatnonzero(relative_edge < .01),
        reconstruction_interval_only=True))

def run_series(series):
    y = np.asarray(DATA[series],float)
    spec = case_spec(series)
    path = OUT / (series+"_fit_new_bounds.json")
    record = load_checkpoint(path,spec,y)
    resumed_fit = record is not None and record.get("fit") is not None
    if not resumed_fit:
        print("START REAL FIT:",series,"| K =",K_VARIABLE,"| seeds =",SEEDS,flush=True)
        started = time.perf_counter()
        try:
            # DATA -> fit_model -> least_squares -> newly optimized theta.
            fit = fit_model(y,**FIT_OPTIONS)
        except NUMERICAL_ERRORS as exc:
            record = dict(spec=spec,stage="fit_failed",fit=None,evaluation=None,
                          runtime=float(time.perf_counter()-started),evaluation_seconds=0.0,
                          fit_error=type(exc).__name__+": "+str(exc))
            save_checkpoint(path,record)
            print("FIT FAILED:",series,record["fit_error"],flush=True)
            print("FINAL PARAMETERS\na = unavailable\nb = unavailable\nRMSE = unavailable",flush=True)
            return record,"fresh_fit_failed"
        elapsed = float(time.perf_counter()-started)
        selected_attempt(fit,y)
        record = dict(spec=spec,stage="fit_complete",fit=fit,evaluation=None,
                      runtime=elapsed,evaluation_seconds=0.0,fit_error=None)
        save_checkpoint(path,record)
        print("FIT SAVED:",series,"| checking finer grids",flush=True)
    else:
        print("RESUME COMPATIBLE NEW-RUN CHECKPOINT:",series,flush=True)
    if record.get("evaluation") is None:
        started = time.perf_counter()
        try:
            record["evaluation"] = evaluate_refit(record["fit"],y)
            record["evaluation_error"] = None
        except NUMERICAL_ERRORS as exc:
            record["evaluation_error"] = type(exc).__name__+": "+str(exc)
        record["evaluation_seconds"] = float(time.perf_counter()-started)
        record["stage"] = "complete" if record.get("evaluation") is not None else "evaluation_failed"
        save_checkpoint(path,record)
    fit,ev = record["fit"],record.get("evaluation")
    print("FINAL PARAMETERS",series,flush=True)
    print("a =",fit["theta"][0],flush=True)
    print("b =",fit["theta"][1],flush=True)
    print("RMSE =",ev["metrics"]["rmse"] if ev is not None else "unavailable: evaluation failed",flush=True)
    print("cost =",fit["selected_cost"],"success =",fit["selected_success"],
          "valid =",selected_attempt(fit,y)["valid"],flush=True)
    return record,"resumed_fit" if resumed_fit else "fresh_fit"

SUMMARY_COLUMNS = ["series","a","b","x0","z0","RMSE","cost","q_min","q_max","q_mean","q_std","success","valid","runtime"]

def make_summary_row(series,record):
    row = dict.fromkeys(SUMMARY_COLUMNS)
    row.update(series=series,success=False,valid=False,runtime=record.get("runtime"))
    fit,ev = record.get("fit"),record.get("evaluation")
    if fit is None: return row
    best = selected_attempt(fit,np.asarray(DATA[series],float))
    q = basis_matrix(fit["n_train"],fit["k"],1,fit["basis_kind"]) @ np.asarray(fit["theta"][5:])
    row.update(a=fit["theta"][0],b=fit["theta"][1],x0=fit["theta"][2],z0=fit["theta"][3],
               RMSE=ev["metrics"]["rmse"] if ev is not None else None,cost=fit["selected_cost"],
               q_min=float(q.min()),q_max=float(q.max()),q_mean=float(q.mean()),q_std=float(q.std()),
               success=bool(fit["selected_success"]),valid=bool(best["valid"]))
    return row

def make_diagnostic_row(series,record):
    fit,ev = record.get("fit") or {},record.get("evaluation") or {}
    mesh = ev.get("mesh") or {}
    flags = []
    if not fit: flags.append("fit_failure")
    if fit and not fit["selected_success"]: flags.append("optimizer_unresolved")
    if fit and not ev: flags.append("evaluation_failure")
    if ev and not mesh.get("grid_pass"): flags.append("mesh_unresolved")
    th = fit.get("theta")
    return dict(series=series,status="+".join(flags) if flags else "numerically_checked",
        h=float(np.exp(th[4])) if th is not None else None,log_h=th[4] if th is not None else None,
        n_parameters=fit.get("n_parameters"),n_samples=len(DATA[series]),
        optimality=fit.get("optimality"),selected_message=fit.get("selected_message"),
        rejected_evaluations=fit.get("rejected_evaluations"),
        n_valid_starts=sum(a["valid"] for a in fit.get("attempts",[])),
        rmse_fit_grid=ev.get("rmse_fit_grid"),rmse_excluding_initial=ev.get("rmse_excluding_initial"),
        rmse_grid_source=ev.get("rmse_grid_source"),reported_substeps=mesh.get("reported_substeps"),
        verification_substeps=mesh.get("verification_substeps"),mesh_pass=bool(mesh.get("grid_pass",False)),
        relative_grid_rmse=mesh.get("relative_grid_rmse"),residual_max=mesh.get("residual_max"),
        near_bound_indices=json.dumps(ev.get("near_bound_indices",[])),
        evaluation_seconds=record.get("evaluation_seconds"),
        error=record.get("fit_error") or record.get("evaluation_error") or ev.get("mesh_error"))

In [ ]:
RUN_STARTED_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat()
records, sources, rows = {}, {}, []
for series in SERIES_NAMES:
    record,origin = run_series(series)
    records[series],sources[series] = record,origin
    rows.append(make_summary_row(series,record))
    pd.DataFrame(rows,columns=SUMMARY_COLUMNS).to_csv(OUT/"fit_summary_new_bounds.csv",index=False)
    display(pd.DataFrame([rows[-1]],columns=SUMMARY_COLUMNS))
summary = pd.DataFrame(rows,columns=SUMMARY_COLUMNS)
diagnostics = pd.DataFrame([make_diagnostic_row(s,records[s]) for s in SERIES_NAMES])
print(f"ALL {len(SERIES_NAMES)} SERIES PROCESSED. Check success, valid and mesh status separately.")
display(summary)
display(diagnostics)

In [ ]:
summary.to_csv(OUT/"fit_summary_new_bounds.csv",index=False)
diagnostics.to_csv(OUT/"fit_diagnostics_new_bounds.csv",index=False)
attempt_rows = []
for series in SERIES_NAMES:
    fit = records[series].get("fit") or {}
    for attempt in fit.get("attempts",[]):
        r = {key:attempt.get(key) for key in ("label","success","valid","cost","optimality","nfev","message")}
        r.update(series=series,a_start=attempt["initial"][0],b_start=attempt["initial"][1])
        attempt_rows.append(r)
    ev = records[series].get("evaluation")
    if ev is not None:
        frame = pd.DataFrame(dict(sample=np.arange(len(DATA[series])),observed=DATA[series],
                         reconstruction=ev["prediction"],q=ev["q"],fit_grid_reconstruction=ev["fit_grid_prediction"]))
        if ev.get("verification_prediction") is not None:
            frame["verification_reconstruction"] = ev["verification_prediction"]
        frame.to_csv(OUT/(series+"_reconstruction_new_bounds.csv"),index=False)
pd.DataFrame(attempt_rows,columns=["series","label","a_start","b_start","success","valid","cost","optimality","nfev","message"]).to_csv(OUT/"multistart_attempts_new_bounds.csv",index=False)

fig,axes = plt.subplots(len(SERIES_NAMES),2,figsize=(14,3*len(SERIES_NAMES)),squeeze=False)
for i,series in enumerate(SERIES_NAMES):
    y = np.asarray(DATA[series],float)
    ev = records[series].get("evaluation")
    status = diagnostics.loc[diagnostics["series"]==series,"status"].iloc[0]
    axes[i,0].plot(np.arange(len(y)),y,color="black",lw=.9,label="observed")
    if ev is not None:
        axes[i,0].plot(ev["prediction"],color="tab:blue",lw=1,label="free-running reconstruction")
        axes[i,1].plot(ev["q"],color="tab:orange",lw=1.1)
    else:
        axes[i,1].text(.5,.5,"No evaluated trajectory",ha="center",va="center",transform=axes[i,1].transAxes)
    axes[i,0].set_title(series+" | "+status,fontsize=10)
    axes[i,0].set_ylabel("QIDS score")
    axes[i,0].legend(fontsize=8)
    axes[i,1].set_title(series+" | fitted effective order")
    axes[i,1].set_ylim(0,1)
    axes[i,1].set_ylabel("q(t)")
    for ax in axes[i]: ax.grid(alpha=.2); ax.set_xlabel("sample index")
fig.tight_layout()
fig.savefig(OUT/"reconstruction_and_q_new_bounds.png",dpi=180)
display(fig)
plt.close(fig)

In [ ]:
checks = dict(
    all_series_processed=(set(records)==set(SERIES_NAMES)),
    series_names=list(SERIES_NAMES), n_series=len(SERIES_NAMES),
    n_calibration_points=int(N_CALIBRATION_POINTS),
    full_lengths=FULL_LENGTHS, calibration_lengths=CALIBRATION_LENGTHS,
    post_calibration_lengths=POST_CALIBRATION_LENGTHS,
    calibration_is_exact_prefix=all(DATA[s] == DATA_FULL[s][:N_CALIBRATION_POINTS]
                                    and len(DATA[s]) == N_CALIBRATION_POINTS for s in SERIES_NAMES),
    fresh_fit_series=[s for s in SERIES_NAMES if sources[s].startswith("fresh_fit")],
    resumed_fit_series=[s for s in SERIES_NAMES if sources[s]=="resumed_fit"],
    n_valid_fits=int(summary["valid"].sum()),
    n_selected_optimizer_success=int(summary["success"].sum()),
    n_mesh_pass=int(diagnostics["mesh_pass"].sum()),
    n_numerically_checked=int((diagnostics["status"]=="numerically_checked").sum()),
    all_valid_fits_have_53_free_parameters=all(
        r["fit"]["free_indices"]==list(range(53)) for r in records.values() if r.get("fit") is not None),
    all_evaluations_on_observed_interval=all(
        r["evaluation"]["reconstruction_interval_only"] for r in records.values() if r.get("evaluation") is not None),
    a_bounds=[-10.0,10.0],b_bounds=[-10.0,10.0],K_VARIABLE=K_VARIABLE)
checks["all_series_numerically_checked"] = checks["n_numerically_checked"] == len(SERIES_NAMES)
atomic_json(OUT/"protocol_checks.json",checks)
atomic_json(OUT/"execution_manifest.json",dict(protocol=PROTOCOL_VERSION,
    run_started_utc=RUN_STARTED_UTC,run_finished_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
    run_status="all_series_processed_in_user_kernel",sources=sources,environment=ENVIRONMENT,
    fit_options=FIT_OPTIONS,source_provenance=SOURCE_PROVENANCE,core_runtime_hash=CORE_RUNTIME_HASH,
    checks=checks))
print(json.dumps(checks,indent=2,ensure_ascii=False))
if not checks["all_series_numerically_checked"]:
    print("Some fits remain unresolved. See fit_diagnostics_new_bounds.csv; no superiority claim follows from completion alone.")
archive = shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print("Results folder:",OUT.resolve())
print("Results archive:",archive)
display(FileLink(archive))

In [ ]:
from scipy.signal import place_poles
C = np.array([0., 1., 0.])
QMIN_OBS, QMAX_OBS = 0.03, 0.98
OBSERVER_INITIAL_HIDDEN = np.array([0.0, 0.0], dtype=float)  # normalized x0,z0 guess
OBSERVER_Q0 = 0.505
OBSERVER_POLE_SCALE = 1.0
OBSERVER_GAMMA_PER_TIME = 0.05 / 0.12
OBSERVER_MAX_Q_RATE_PER_TIME = 1.0 / 21.48
OBSERVER_SENSITIVITY_EPSILON = 1e-8
OBSERVER_POLES = np.array([-1., -2., -3.])
OBSERVER_SOLVER = Config(newton_tol=1e-10, max_newton=35, state_limit=1e4)
assert OBSERVER_INITIAL_HIDDEN.shape == (2,)
assert QMIN_OBS == .03 and QMAX_OBS == .98
assert QMIN_OBS <= OBSERVER_Q0 <= QMAX_OBS

In [ ]:
def design_gain_online(y0, a, b, pole_scale=1.0):
    if not np.isfinite(pole_scale) or pole_scale < 0:
        raise ValueError("Invalid pole scale")
    if pole_scale == 0:
        return np.zeros(3)
    nominal = np.array([OBSERVER_INITIAL_HIDDEN[0], y0, OBSERVER_INITIAL_HIDDEN[1]])
    A = jacobian(nominal, a, b)
    observability = np.vstack([C, C @ A, C @ A @ A])
    if np.linalg.matrix_rank(observability) < 3:
        raise NumericalFailure("Nominal linearization is not observable")
    poles = pole_scale * OBSERVER_POLES
    L = place_poles(A.T, C[:, None], poles).gain_matrix.T.ravel()
    eigs = np.sort_complex(np.linalg.eigvals(A - np.outer(L, C)))
    if not np.allclose(eigs, np.sort_complex(poles), rtol=1e-6, atol=1e-8):
        raise NumericalFailure("Pole placement verification failed")
    return L

def implicit_observer_step_online(history, q, h, a, b, measured_y, gain, cfg=OBSERVER_SOLVER):
    hist = np.asarray(history, dtype=float)
    gain = np.asarray(gain, dtype=float)
    if hist.ndim != 2 or hist.shape[1] != 3 or len(hist) < 1 or not np.all(np.isfinite(hist)):
        raise ValueError("A finite state history is required")
    if gain.shape != (3,) or not np.all(np.isfinite(gain)) or not np.isfinite(measured_y):
        raise ValueError("Invalid gain or observation")
    if not np.isfinite(q) or not QMIN_OBS <= q <= QMAX_OBS:
        raise ValueError("Order outside configured bounds")
    if np.all(gain == 0):
        return implicit_step(hist, q, h, a, b, cfg=cfg)
    weights, dweights, c, dc = l1_kernel(q, len(hist), h)
    differences = np.diff(hist, axis=0)[::-1]
    rhs = hist[-1] - weights @ differences
    def residual(x):
        return c * (x - rhs) - field(x, a, b) - gain * (measured_y - C @ x)
    x = hist[-1].copy(); r = residual(x)
    for iteration in range(cfg.max_newton + 1):
        norm = float(np.linalg.norm(r, np.inf))
        if np.isfinite(norm) and norm <= cfg.newton_tol:
            break
        if iteration == cfg.max_newton or not np.isfinite(norm):
            raise NumericalFailure("Observer Newton residual did not converge")
        matrix = c * np.eye(3) - jacobian(x, a, b) + np.outer(gain, C)
        try:
            step = np.linalg.solve(matrix, -r)
        except np.linalg.LinAlgError as exc:
            raise NumericalFailure("Singular observer Newton matrix") from exc
        accepted = False
        for power in range(22):
            candidate = x + (0.5 ** power) * step
            if not np.all(np.isfinite(candidate)) or np.max(np.abs(candidate)) > cfg.state_limit:
                continue
            rr = residual(candidate)
            if np.all(np.isfinite(rr)) and np.linalg.norm(rr, np.inf) < norm:
                x, r, accepted = candidate, rr, True
                break
        if not accepted:
            raise NumericalFailure("Observer Newton line search failed")
    if not np.all(np.isfinite(x)) or np.max(np.abs(x)) > cfg.state_limit:
        raise NumericalFailure("Observer state rejected")
    partial = dc * (x - rhs) + c * (dweights @ differences)
    try:
        sensitivity = np.linalg.solve(c * np.eye(3) - jacobian(x, a, b) + np.outer(gain, C), -partial)
    except np.linalg.LinAlgError as exc:
        raise NumericalFailure("Singular local order-sensitivity matrix") from exc
    if not np.all(np.isfinite(sensitivity)):
        raise NumericalFailure("Nonfinite local order sensitivity")
    return x, sensitivity, float(np.linalg.norm(r, np.inf)), iteration

In [ ]:
class StreamingAdaptiveObserver:
    """Causal state/order observer; update consumes exactly one new y value."""
    def __init__(self, y0, a, b, h, initial_hidden, q0, pole_scale,
                 gamma_per_time, max_q_rate, sensitivity_epsilon, mode="adaptive"):
        if mode not in ("adaptive", "fixed_q"):
            raise ValueError("Only adaptive/fixed_q modes are allowed for real data")
        hidden = np.asarray(initial_hidden, dtype=float)
        vals = np.r_[y0, a, b, h, hidden, q0, pole_scale, gamma_per_time, max_q_rate, sensitivity_epsilon]
        if hidden.shape != (2,) or not np.all(np.isfinite(vals)) or h <= 0:
            raise ValueError("Invalid stream initialization")
        if not QMIN_OBS <= q0 <= QMAX_OBS or gamma_per_time < 0 or max_q_rate <= 0 or sensitivity_epsilon <= 0:
            raise ValueError("Invalid order adaptation settings")
        self.a, self.b, self.h = float(a), float(b), float(h)
        self.gain = design_gain_online(float(y0), self.a, self.b, float(pole_scale))
        self.q = float(q0); self.mode = mode
        self.gamma = float(gamma_per_time); self.max_q_rate = float(max_q_rate); self.eps = float(sensitivity_epsilon)
        self.history = [np.array([hidden[0], y0, hidden[1]], dtype=float)]
        self.rows = [dict(sample=0, x_hat=float(hidden[0]), y_hat_normalized=float(y0),
                          z_hat=float(hidden[1]), q_used=float(q0), q_after=float(q0),
                          innovation_normalized=0.0, local_sensitivity=np.nan, raw_rate=np.nan,
                          newton_residual=np.nan, newton_iterations=0,
                          rate_limited=False, boundary_limited=False)]
    def update(self, measured_y):
        measured_y = float(measured_y)
        if not np.isfinite(measured_y):
            raise ValueError("Current observation must be finite")
        q_used = self.q
        state, sensitivity, residual, iterations = implicit_observer_step_online(
            np.asarray(self.history), q_used, self.h, self.a, self.b,
            measured_y, self.gain, cfg=OBSERVER_SOLVER)
        innovation = measured_y - C @ state
        q_after, raw_rate = q_used, np.nan
        rate_limited = boundary_limited = False
        if self.mode == "adaptive":
            raw_rate = self.gamma * innovation * sensitivity[1] / (self.eps + sensitivity[1] ** 2)
            if not np.isfinite(raw_rate):
                raise NumericalFailure("Nonfinite adaptive q rate")
            limited_rate = float(np.clip(raw_rate, -self.max_q_rate, self.max_q_rate))
            candidate = q_used + self.h * limited_rate
            q_after = float(np.clip(candidate, QMIN_OBS, QMAX_OBS))
            rate_limited = bool(abs(raw_rate) > self.max_q_rate)
            boundary_limited = bool(candidate < QMIN_OBS or candidate > QMAX_OBS)
        if not np.all(np.isfinite(state)) or not QMIN_OBS <= q_after <= QMAX_OBS:
            raise NumericalFailure("Invalid observer update")
        row = dict(sample=len(self.history), x_hat=float(state[0]), y_hat_normalized=float(state[1]),
                   z_hat=float(state[2]), q_used=float(q_used), q_after=float(q_after),
                   innovation_normalized=float(innovation), local_sensitivity=float(sensitivity[1]),
                   raw_rate=float(raw_rate), newton_residual=float(residual),
                   newton_iterations=int(iterations), rate_limited=rate_limited,
                   boundary_limited=boundary_limited)
        self.q = q_after
        self.history.append(state.copy())
        self.rows.append(row)
        return row

def observer_calibration_input(series, record, full_y, n_calibration):
    fit = record.get("fit")
    if fit is None:
        raise ValueError("Stage-one calibration failed")
    prefix = np.asarray(full_y[:n_calibration], dtype=float)
    if fit["n_train"] != n_calibration or fit["train_hash"] != fingerprint(prefix.tolist()):
        raise ValueError("Calibration prefix does not match stage-one data")
    mu, sd = float(fit["mu"]), float(fit["sd"])
    if not np.isclose(mu, prefix.mean(), rtol=0, atol=1e-12) or not np.isclose(sd, prefix.std(), rtol=0, atol=1e-12):
        raise ValueError("Stage-one normalization is not prefix-only")
    a, b, log_h = [float(fit["theta"][j]) for j in (0, 1, 4)]
    h = float(np.exp(log_h))
    mesh = (record.get("evaluation") or {}).get("mesh") or {}
    return dict(series=series, a=a, b=b, h=h, mu=mu, sd=sd,
                n_calibration=int(n_calibration), prefix_hash=fit["train_hash"],
                selected_cost=float(fit["selected_cost"]),
                calibration_success=bool(fit["selected_success"]),
                calibration_mesh_pass=bool(mesh.get("grid_pass", False)))

def observe_series_online(full_y, calibration, mode="adaptive"):
    """Start local observer time at global index n_calibration-1, without prefix replay.

    Global index n-1 is observer sample zero; only global n..N-1 are passed
    to stream.update(). The observer's L1 memory begins at global n-1.
    """
    full_y = np.asarray(full_y, dtype=float)
    n = int(calibration["n_calibration"])
    if (full_y.ndim != 1 or n < 2 or len(full_y) < n
            or not np.all(np.isfinite(full_y[:n]))):
        raise ValueError("Full series must include its complete finite calibration prefix")
    if fingerprint(full_y[:n].tolist()) != calibration["prefix_hash"]:
        raise ValueError("Observer input does not match the fitted calibration prefix")
    start = n - 1
    y0 = (float(full_y[start]) - calibration["mu"]) / calibration["sd"]
    stream = StreamingAdaptiveObserver(
        y0, calibration["a"], calibration["b"], calibration["h"],
        OBSERVER_INITIAL_HIDDEN, OBSERVER_Q0, OBSERVER_POLE_SCALE,
        OBSERVER_GAMMA_PER_TIME, OBSERVER_MAX_Q_RATE_PER_TIME,
        OBSERVER_SENSITIVITY_EPSILON, mode=mode)
    failure = None
    started = time.perf_counter()
    # Do NOT use full_y[:start] and do NOT feed full_y[start] to update().
    for value in full_y[n:]:
        try:
            stream.update((float(value) - calibration["mu"]) / calibration["sd"])
        except (NumericalFailure, FloatingPointError, np.linalg.LinAlgError, OverflowError, ValueError) as exc:
            failure = f"{type(exc).__name__}: {exc}"
            break
    # Retain full-record sample numbers without manufacturing past observer states.
    frame = pd.DataFrame(stream.rows)
    frame["observer_sample"] = frame["sample"]  # local 0 corresponds to global n-1
    frame["sample"] = frame["sample"] + start
    frame = frame.set_index("sample").reindex(range(len(full_y))).rename_axis("sample").reset_index()
    frame["observed_y"] = full_y
    frame["y_hat"] = frame["y_hat_normalized"] * calibration["sd"] + calibration["mu"]
    frame["model_time"] = (frame["sample"] - start) * calibration["h"]
    frame.loc[frame["sample"] < start, "model_time"] = np.nan
    frame["calibration_prefix"] = frame["sample"] < n
    frame["observer_initialization"] = frame["sample"] == start
    frame["computed"] = (frame["sample"] >= start) & (frame["sample"] < start + len(stream.rows))
    return frame, dict(
        status="complete" if failure is None else "numerical_failure", error=failure,
        n_completed=len(stream.rows), n_total=len(full_y)-start, n_full_record=len(full_y),
        global_start_index=start, first_update_index=n, prefix_replayed=False,
        memory_origin_global_index=start, gain=stream.gain.tolist(),
        runtime_seconds=float(time.perf_counter()-started))

def online_row(series, frame, meta, cal):
    start = int(cal["n_calibration"])-1
    done = frame.loc[frame["computed"]]
    active = done.loc[done["sample"] > start]
    tail = active.loc[active["sample"] >= cal["n_calibration"]]
    def rmse(p):
        return float(np.sqrt(np.mean((p["observed_y"] - p["y_hat"])**2))) if len(p) else None
    return json_ready(dict(series=series, **meta, a=cal["a"], b=cal["b"], h=cal["h"],
        n_calibration=cal["n_calibration"], n_full=len(frame), tail_count=len(frame)-cal["n_calibration"],
        calibration_success=cal["calibration_success"], calibration_mesh_pass=cal["calibration_mesh_pass"],
        q0=OBSERVER_Q0, q_used_min=float(done["q_used"].min()), q_used_max=float(done["q_used"].max()),
        q_after_last=float(done["q_after"].iloc[-1]), q_used_mean=float(done["q_used"].mean()),
        q_used_std=float(done["q_used"].std(ddof=0)),
        y_tracking_rmse=rmse(active), y_tracking_rmse_tail=rmse(tail),
        max_newton_residual=float(active["newton_residual"].max()) if len(active) else None,
        rate_limit_fraction=float(active["rate_limited"].mean()) if len(active) else None,
        boundary_limit_fraction=float(active["boundary_limited"].mean()) if len(active) else None,
        max_abs_x=float(done["x_hat"].abs().max()), max_abs_z=float(done["z_hat"].abs().max()),
        inference_note="post-prefix causal tracking after starting observer memory at global sample n-1; no prefix replay"))


### اجرای مرحلهٔ دوم و آزمون علیت

پس از اجرای این سلول، برای هر سری فایل CSV مسیر تخمین و یک checkpoint JSON نوشته می‌شود. اگر اجرای سلول قطع شود، برازش بخش اول از checkpoint همان نوت‌بوک ادامه می‌یابد؛ checkpoint مرحلهٔ دوم نیز جداگانه نگه داشته می‌شود.

In [ ]:
ONLINE_OUT = OUT / f"online_observer_start{N_CALIBRATION_POINTS-1}"
ONLINE_OUT.mkdir(parents=True, exist_ok=True)
ONLINE_RECORDS, ONLINE_SUMMARY = {}, []
for series in SERIES_NAMES:
    try:
        fit_before_observation = fingerprint(records[series]["fit"])
        calibration = observer_calibration_input(series, records[series], DATA_FULL[series], N_CALIBRATION_POINTS)
        fit = records[series]["fit"]
        assert calibration["a"] == fit["theta"][0] and calibration["b"] == fit["theta"][1]
        assert calibration["h"] == float(np.exp(fit["theta"][4]))
        assert calibration["mu"] == fit["mu"] and calibration["sd"] == fit["sd"]
        frame, meta = observe_series_online(DATA_FULL[series], calibration, mode="adaptive")
        if meta["status"] != "complete":
            raise NumericalFailure(meta["error"])
        # Conditional on fixed calibration, updates must not depend on future samples.
        m = min(30, len(DATA_FULL[series])-N_CALIBRATION_POINTS)
        short_end = N_CALIBRATION_POINTS + m
        short_frame, short_meta = observe_series_online(DATA_FULL[series][:short_end], calibration, mode="adaptive")
        cols = ["x_hat", "y_hat_normalized", "z_hat", "q_used", "q_after", "innovation_normalized"]
        for col in cols:
            assert np.allclose(frame.loc[N_CALIBRATION_POINTS-1:short_end-1, col],
                               short_frame.loc[N_CALIBRATION_POINTS-1:short_end-1, col],
                               rtol=0, atol=1e-12, equal_nan=True)
        start = N_CALIBRATION_POINTS - 1
        assert meta["global_start_index"] == start and meta["memory_origin_global_index"] == start
        assert meta["first_update_index"] == N_CALIBRATION_POINTS and not meta["prefix_replayed"]
        assert meta["n_completed"] == len(DATA_FULL[series]) - start
        assert frame.loc[:start-1, "computed"].eq(False).all()
        assert frame.loc[:start-1, "y_hat"].isna().all()
        assert bool(frame.loc[start, "observer_initialization"])
        assert frame.loc[start, "observer_sample"] == 0
        assert np.isclose(frame.loc[start, "y_hat"], DATA_FULL[series][start], rtol=0, atol=1e-12)
        assert np.isclose(frame.loc[start, "x_hat"], OBSERVER_INITIAL_HIDDEN[0], atol=1e-12)
        assert np.isclose(frame.loc[start, "z_hat"], OBSERVER_INITIAL_HIDDEN[1], atol=1e-12)
        assert frame.loc[start+1, "observer_sample"] == 1
        assert calibration["prefix_hash"] == fingerprint(list(np.asarray(DATA_FULL[series][:N_CALIBRATION_POINTS], float)))
        assert fingerprint(records[series]["fit"]) == fit_before_observation, "Calibration was modified"
        frame.to_csv(ONLINE_OUT / f"{series}_online_observer.csv", index=False)
        record = dict(series=series, calibration=calibration, meta=meta,
                      summary=online_row(series, frame, meta, calibration))
        atomic_json(ONLINE_OUT / f"{series}_online_observer.json", record)
        ONLINE_RECORDS[series] = record; ONLINE_SUMMARY.append(record["summary"])
        print("ONLINE OBSERVER COMPLETE:", series, "| a =", calibration["a"], "| b =", calibration["b"],
              "| q range =", (frame["q_used"].min(), frame["q_used"].max()),
              "| post-calibration y RMSE =", record["summary"]["y_tracking_rmse_tail"])
    except (NumericalFailure, ValueError, FloatingPointError, np.linalg.LinAlgError, OverflowError) as exc:
        print("ONLINE OBSERVER FAILED:", series, type(exc).__name__, str(exc))
        ONLINE_RECORDS[series] = dict(series=series, status="failed", error=f"{type(exc).__name__}: {exc}")

online_summary = pd.DataFrame(ONLINE_SUMMARY)
online_summary.to_csv(ONLINE_OUT / "online_observer_summary.csv", index=False)
assert set(ONLINE_RECORDS) == set(SERIES_NAMES)
display(online_summary)

In [ ]:
# Observer update audit, conditional on fixed prefix calibration, and diagnostic figure.
try:
    source_online = inspect.getsource(observe_series_online)
except (OSError, TypeError):
    source_online = ""
if source_online:
    assert "fit_model" not in source_online and "least_squares" not in source_online
    assert "DATA_FULL.mean" not in source_online and "DATA_FULL.std" not in source_online
assert all(CALIBRATION_LENGTHS[s] == N_CALIBRATION_POINTS for s in SERIES_NAMES)
assert all(DATA[s] == DATA_FULL[s][:N_CALIBRATION_POINTS] for s in SERIES_NAMES)
assert all(POST_CALIBRATION_LENGTHS[s] == FULL_LENGTHS[s] - N_CALIBRATION_POINTS for s in SERIES_NAMES)
assert all(r.get("meta",{}).get("status") == "complete" for r in ONLINE_RECORDS.values())

fig, axes = plt.subplots(len(SERIES_NAMES), 2, figsize=(14, 3*len(SERIES_NAMES)), squeeze=False)
for i, series in enumerate(SERIES_NAMES):
    frame = pd.read_csv(ONLINE_OUT / f"{series}_online_observer.csv")
    y = frame["observed_y"].to_numpy()
    axes[i,0].plot(y, color="black", lw=.8, label="observed QIDS (y)")
    axes[i,0].plot(frame["y_hat"], color="tab:blue", lw=1, label="online y estimate")
    axes[i,0].axvline(CALIBRATION_LENGTHS[series]-1, color="tab:red", ls=":", lw=.9, label="calibration boundary")
    axes[i,0].set_title(series + " | output tracking")
    axes[i,0].legend(fontsize=7); axes[i,0].grid(alpha=.2)
    axes[i,1].plot(frame["q_after"], color="tab:orange", lw=1, label="adaptive q")
    axes[i,1].axhline(QMIN_OBS, color="0.5", ls=":"); axes[i,1].axhline(QMAX_OBS, color="0.5", ls=":")
    axes[i,1].axvline(CALIBRATION_LENGTHS[series]-1, color="tab:red", ls=":", lw=.9)
    axes[i,1].set_ylim(0, 1); axes[i,1].set_title(series + " | causal adaptive q")
    axes[i,1].legend(fontsize=7); axes[i,1].grid(alpha=.2)
for ax in axes[-1]: ax.set_xlabel("sample index")
fig.tight_layout(); fig.savefig(ONLINE_OUT / "online_observer_diagnostics.png", dpi=180)
display(fig); plt.close(fig)
checks_online = dict(
    full_data_used_in_stage_two=False,
    original_sample_indexing_preserved=True,
    observer_updates_only_on_post_calibration_samples=True,
    stage_one_uses_prefix_only=True,
    n_calibration_points=int(N_CALIBRATION_POINTS),
    calibration_lengths=CALIBRATION_LENGTHS,
    post_calibration_lengths=POST_CALIBRATION_LENGTHS,
    series_names=list(SERIES_NAMES), n_series=len(SERIES_NAMES),
    all_series_complete=all(r.get("meta",{}).get("status") == "complete" for r in ONLINE_RECORDS.values()),
    causal_prefix_checks=True,
    causality_condition="fixed calibration on indices 0..59; observer initialized at 59; updates at 60..N-1",
    observer_start_global_index=N_CALIBRATION_POINTS-1,
    first_update_global_index=N_CALIBRATION_POINTS,
    observer_memory_origin=N_CALIBRATION_POINTS-1,
    prefix_replayed=False,
    observer_init_y_from_last_stage_one_sample=True,
    adaptive_mode=True,
    future_measurements_not_passed_to_update=True,
    q_bounds=[QMIN_OBS, QMAX_OBS],
    output_space="y is reported in original units; x and z are model-normalized coordinates",
)
atomic_json(ONLINE_OUT / "online_protocol_checks.json", checks_online)
print(json.dumps(checks_online, indent=2, ensure_ascii=False))
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Combined results archive:", archive)
display(FileLink(archive))